# Niching-Based Evolutionary Feature Selection for Wild Animal Sound Classification (MLP Version)

Mercy Doan (s1180494), Batoul Hammoud (s1180496), Ian Kahn (s11804)

This notebook implements a feature selection pipeline for wildlife audio classification on the BirdCLEF+ 2026 dataset using **MLP (Multi-Layer Perceptron)** as the inner classifier instead of Random Forest.

The notebook is structured into five stages following the proposal. Data preparation, feature extraction with the librosa library, baseline classifiers, evolutionary search with fitness sharing, and evaluation across seeds and niching thresholds.

## Setup

This section sets up library installations, file paths, and the project's pipeline modules. The five Python files implementing the pipeline (`data.py`, `features.py`, `baselines.py`, `ec.py`, and `evaluate.py`) are written to disk below as a self-contained `src` package, so that subsequent cells can import from them. The BirdCLEF+ 2026 dataset is mounted automatically by Kaggle as a competition data source.

### MLP Configuration
We use a small single-hidden-layer MLP with L-BFGS optimizer and StandardScaler in front, since MLPs need scaled inputs.

In [3]:
!pip install -q mrmr-selection

In [4]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# MLP imports
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORK = Path("/kaggle/working")
DATA_DIR = WORK / "data"
RESULTS_DIR = WORK / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(WORK / "src").mkdir(parents=True, exist_ok=True)

def _find_data_dir():
    # Probe pattern from the kaggle-cli skill: handle both
    # /kaggle/input/<slug>/ and /kaggle/input/competitions/<slug>/
    # by looking for train.csv at depth 1 then depth 2.
    root = Path("/kaggle/input")
    if root.exists():
        for p in sorted(root.iterdir()):
            if (p / "train.csv").exists():
                return p
        for p in root.iterdir():
            if p.is_dir():
                for sub in p.iterdir():
                    if (sub / "train.csv").exists():
                        return sub
    raise FileNotFoundError("BirdCLEF train.csv not found under /kaggle/input")

RAW_DATA_DIR = _find_data_dir()

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

# Fixed seed for the train/val/test split. The GA seeds are set in Stage 4.
MANIFEST_SEED = 42

# MLP Hyperparameters
MLP_HIDDEN = (16,)
MLP_MAX_ITER = 200
MLP_SOLVER = "adam"

print(f"WORK         = {WORK}")
print(f"RAW_DATA_DIR = {RAW_DATA_DIR}")
print(f"RAW_DATA_DIR exists: {RAW_DATA_DIR.exists()}")
print(f"MLP Configuration:")
print(f"  hidden_layer_sizes = {MLP_HIDDEN}")
print(f"  solver = {MLP_SOLVER}")
print(f"  max_iter = {MLP_MAX_ITER}")

WORK         = /kaggle/working
RAW_DATA_DIR = /kaggle/input/competitions/birdclef-2026
RAW_DATA_DIR exists: True
MLP Configuration:
  hidden_layer_sizes = (16,)
  solver = adam
  max_iter = 200


### Pipeline modules

The pipeline logic is split across five Python files, each implementing one stage. The cells below write these files to disk as a self-contained package.

In [5]:
%%writefile src/__init__.py
# Package marker so `from src import ...` works on Kaggle.

Writing src/__init__.py


In [6]:
%%writefile src/data.py
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

MANIFEST_COLUMNS = [
    "clip_id",
    "primary_label",
    "common_name",
    "class_name",
    "filename",
    "split",
]


def build_manifest(
    raw_data_dir: Path,
    species_specs: list[tuple[str, int]],
    split_ratios: tuple[float, float, float] = (0.70, 0.15, 0.15),
    seed: int = 42,
) -> pd.DataFrame:
    if not np.isclose(sum(split_ratios), 1.0):
        raise ValueError(f"split_ratios must sum to 1.0, got {split_ratios}")

    train_csv = pd.read_csv(Path(raw_data_dir) / "train.csv")
    rng = np.random.default_rng(seed)

    parts: list[pd.DataFrame] = []
    for label, target in species_specs:
        species_df = train_csv[train_csv["primary_label"] == label].reset_index(drop=True)
        if len(species_df) == 0:
            raise ValueError(f"No clips found for primary_label '{label}'")
        if len(species_df) < target:
            raise ValueError(
                f"Species '{label}' has only {len(species_df)} clips, "
                f"requested {target}"
            )

        idx = rng.permutation(len(species_df))[:target]
        sampled = species_df.iloc[idx].reset_index(drop=True)
        sampled["split"] = _assign_splits(target, split_ratios, rng)
        parts.append(sampled)

    manifest = pd.concat(parts, ignore_index=True)
    manifest["clip_id"] = manifest["filename"].apply(lambda f: Path(f).stem)
    return manifest[MANIFEST_COLUMNS]


def _assign_splits(
    n: int,
    ratios: tuple[float, float, float],
    rng: np.random.Generator,
) -> np.ndarray:
    n_train = int(round(n * ratios[0]))
    n_val = int(round(n * ratios[1]))
    n_test = n - n_train - n_val
    labels = np.array(
        ["train"] * n_train + ["val"] * n_val + ["test"] * n_test,
        dtype=object,
    )
    rng.shuffle(labels)
    return labels


def split_counts(manifest: pd.DataFrame) -> pd.DataFrame:
    return (
        manifest.groupby(["primary_label", "split"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["train", "val", "test"], fill_value=0)
    )

Writing src/data.py


In [7]:
%%writefile src/features.py
from __future__ import annotations

import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

DEFAULT_SR = 22050
DEFAULT_WINDOW_SECONDS = 5.0
DEFAULT_N_MFCC = 13
DEFAULT_N_FFT = 2048
DEFAULT_HOP_LENGTH = 512


def feature_columns(n_mfcc: int = DEFAULT_N_MFCC) -> list[str]:
    return (
        [f"mfcc_{i}_mean" for i in range(n_mfcc)]
        + [f"mfcc_{i}_std" for i in range(n_mfcc)]
        + [
            "spectral_centroid_mean",
            "spectral_centroid_std",
            "spectral_rolloff_mean",
            "spectral_rolloff_std",
            "spectral_flux_mean",
            "spectral_flux_std",
            "zcr_mean",
            "zcr_std",
        ]
    )


METADATA_COLUMNS = [
    "window_id",
    "clip_id",
    "primary_label",
    "common_name",
    "class_name",
    "split",
    "window_index",
]


def extract_features_for_clip(
    audio_path: Path,
    sr: int = DEFAULT_SR,
    window_seconds: float = DEFAULT_WINDOW_SECONDS,
    n_mfcc: int = DEFAULT_N_MFCC,
    n_fft: int = DEFAULT_N_FFT,
    hop_length: int = DEFAULT_HOP_LENGTH,
) -> list[dict]:
    y, _ = librosa.load(str(audio_path), sr=sr, mono=True)
    n_samples = int(window_seconds * sr)
    if len(y) < n_samples:
        return []

    out = []
    n_windows = len(y) // n_samples
    for i in range(n_windows):
        chunk = y[i * n_samples : (i + 1) * n_samples]
        feats = _features_for_window(chunk, sr, n_mfcc, n_fft, hop_length)
        feats["window_index"] = i
        out.append(feats)
    return out


def _features_for_window(
    y: np.ndarray, sr: int, n_mfcc: int, n_fft: int, hop_length: int
) -> dict:
    out: dict = {}

    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    for i in range(n_mfcc):
        out[f"mfcc_{i}_mean"] = float(np.mean(mfcc[i]))
        out[f"mfcc_{i}_std"] = float(np.std(mfcc[i]))

    centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length
    )
    out["spectral_centroid_mean"] = float(np.mean(centroid))
    out["spectral_centroid_std"] = float(np.std(centroid))

    rolloff = librosa.feature.spectral_rolloff(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length
    )
    out["spectral_rolloff_mean"] = float(np.mean(rolloff))
    out["spectral_rolloff_std"] = float(np.std(rolloff))

    # Spectral flux: L2 norm of frame-to-frame differences in the magnitude
    # spectrum. librosa has no direct call, so compute from the STFT.
    spectrum = np.abs(librosa.stft(y=y, n_fft=n_fft, hop_length=hop_length))
    flux = np.linalg.norm(np.diff(spectrum, axis=1), axis=0)
    out["spectral_flux_mean"] = float(np.mean(flux))
    out["spectral_flux_std"] = float(np.std(flux))

    zcr = librosa.feature.zero_crossing_rate(
        y=y, frame_length=n_fft, hop_length=hop_length
    )
    out["zcr_mean"] = float(np.mean(zcr))
    out["zcr_std"] = float(np.std(zcr))

    return out


def extract_all(
    manifest: pd.DataFrame,
    raw_data_dir: Path,
    audio_subdir: str = "train_audio",
    progress: bool = True,
    sr: int = DEFAULT_SR,
    window_seconds: float = DEFAULT_WINDOW_SECONDS,
    n_mfcc: int = DEFAULT_N_MFCC,
    n_fft: int = DEFAULT_N_FFT,
    hop_length: int = DEFAULT_HOP_LENGTH,
) -> pd.DataFrame:
    audio_root = Path(raw_data_dir) / audio_subdir
    iterator = list(manifest.itertuples(index=False))

    if progress:
        from tqdm.auto import tqdm

        iterator = tqdm(iterator, desc="Extracting features")

    rows: list[dict] = []
    skipped: list[str] = []
    for row in iterator:
        path = audio_root / row.filename
        try:
            window_feats = extract_features_for_clip(
                path,
                sr=sr,
                window_seconds=window_seconds,
                n_mfcc=n_mfcc,
                n_fft=n_fft,
                hop_length=hop_length,
            )
        except Exception as exc:
            warnings.warn(f"Failed to load {row.filename}: {exc}")
            skipped.append(row.clip_id)
            continue
        if not window_feats:
            skipped.append(row.clip_id)
            continue
        for w in window_feats:
            w["clip_id"] = row.clip_id
            w["primary_label"] = row.primary_label
            w["common_name"] = row.common_name
            w["class_name"] = row.class_name
            w["split"] = row.split
            w["window_id"] = f"{row.clip_id}_w{w['window_index']}"
            rows.append(w)

    if skipped:
        warnings.warn(
            f"Skipped {len(skipped)} clips (shorter than {window_seconds}s "
            f"or failed to load)"
        )

    df = pd.DataFrame(rows)
    return df[METADATA_COLUMNS + feature_columns(n_mfcc)]

Writing src/features.py


In [8]:
%%writefile src/baselines.py
from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

DEFAULT_RF_KWARGS = {"n_estimators": 100, "n_jobs": 4}


def train_rf(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    seed: int = 42,
    **rf_kwargs,
) -> RandomForestClassifier:
    kwargs = {**DEFAULT_RF_KWARGS, "random_state": seed, **rf_kwargs}
    rf = RandomForestClassifier(**kwargs)
    rf.fit(X_train, y_train)
    return rf


def select_features_mrmr(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    k: int,
) -> list[str]:
    from mrmr import mrmr_classif

    return mrmr_classif(X=X_train, y=y_train, K=k, show_progress=False)


def evaluate_predictions(
    y_true: Sequence,
    y_pred: Sequence,
    labels: Sequence,
) -> dict:
    overall = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    per_species = recall_score(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )
    return {
        "accuracy": float(overall),
        "macro_f1": float(macro),
        "per_species_recall": {
            str(label): float(value) for label, value in zip(labels, per_species)
        },
    }


def macro_f1(
    y_true: Sequence,
    y_pred: Sequence,
    labels: Sequence,
) -> float:
    return float(
        f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    )


def split_features(
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
) -> dict:
    out = {}
    for split in ("train", "val", "test"):
        sub = features_df[features_df["split"] == split]
        out[split] = {
            "X": sub[list(feature_cols)],
            "y": sub[label_col],
        }
    return out


def run_baseline_all_features(
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    seed: int = 42,
) -> dict:
    parts = split_features(features_df, feature_cols, label_col)
    rf = train_rf(parts["train"]["X"], parts["train"]["y"], seed=seed)
    y_pred = rf.predict(parts["test"]["X"])
    labels = sorted(features_df[label_col].unique())
    metrics = evaluate_predictions(parts["test"]["y"], y_pred, labels)
    metrics["method"] = "all_features"
    metrics["selected_features"] = list(feature_cols)
    return metrics


def run_baseline_mrmr(
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    k: int = 15,
    seed: int = 42,
) -> dict:
    parts = split_features(features_df, feature_cols, label_col)
    selected = select_features_mrmr(parts["train"]["X"], parts["train"]["y"], k=k)
    rf = train_rf(parts["train"]["X"][selected], parts["train"]["y"], seed=seed)
    y_pred = rf.predict(parts["test"]["X"][selected])
    labels = sorted(features_df[label_col].unique())
    metrics = evaluate_predictions(parts["test"]["y"], y_pred, labels)
    metrics["method"] = f"mrmr_top{k}"
    metrics["selected_features"] = selected
    return metrics

Writing src/baselines.py


In [9]:
%%writefile src/ec.py
from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd

from src.baselines import macro_f1, split_features, train_rf

DEFAULT_POP_SIZE = 50
DEFAULT_N_GENERATIONS = 50
DEFAULT_TOURNAMENT_K = 3
DEFAULT_SHARING_THRESHOLD = 0.7
DEFAULT_TOP_K = 5


def run_ga(
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    pop_size: int = DEFAULT_POP_SIZE,
    n_generations: int = DEFAULT_N_GENERATIONS,
    tournament_k: int = DEFAULT_TOURNAMENT_K,
    mutation_rate: float | None = None,
    sharing: bool = False,
    sharing_threshold: float = DEFAULT_SHARING_THRESHOLD,
    top_k: int = DEFAULT_TOP_K,
    seed: int = 42,
    rf_seed: int = 42,
    progress: bool = True,
) -> dict:
    rng = np.random.default_rng(seed)
    n_features = len(feature_cols)
    if mutation_rate is None:
        mutation_rate = 1.0 / n_features

    parts = split_features(features_df, feature_cols, label_col)
    X_train, y_train = parts["train"]["X"], parts["train"]["y"]
    X_val, y_val = parts["val"]["X"], parts["val"]["y"]
    labels = sorted(features_df[label_col].unique())

    population = _init_population(pop_size, n_features, rng)

    history: list[dict] = []
    raw_fitness = np.zeros(pop_size)

    iterator = range(n_generations)
    if progress:
        from tqdm.auto import tqdm
        iterator = tqdm(iterator, desc="GA generations")

    for gen in iterator:
        raw_fitness = np.array(
            [
                _fitness(
                    population[i], X_train, y_train, X_val, y_val, labels, rf_seed
                )
                for i in range(pop_size)
            ]
        )

        if sharing:
            shared_fitness = apply_fitness_sharing(
                raw_fitness, population, sharing_threshold
            )
        else:
            shared_fitness = raw_fitness

        history.append(
            {
                "generation": gen,
                "best_raw": float(raw_fitness.max()),
                "mean_raw": float(raw_fitness.mean()),
                "best_shared": float(shared_fitness.max()),
                "mean_shared": float(shared_fitness.mean()),
                "diversity": _diversity(population),
            }
        )

        if gen == n_generations - 1:
            break

        new_population = np.empty_like(population)
        for i in range(pop_size):
            p1 = _tournament(shared_fitness, tournament_k, rng)
            p2 = _tournament(shared_fitness, tournament_k, rng)
            child = uniform_crossover(population[p1], population[p2], rng)
            child = bit_flip_mutate(child, mutation_rate, rng)
            if child.sum() == 0:
                child[rng.integers(n_features)] = 1
            new_population[i] = child
        population = new_population

    best_idx = int(np.argmax(raw_fitness))
    top_indices = np.argsort(raw_fitness)[-top_k:][::-1]

    return {
        "history": history,
        "final_population": population,
        "raw_fitness": raw_fitness,
        "best_mask": population[best_idx].copy(),
        "best_fitness": float(raw_fitness[best_idx]),
        "top_k_masks": [population[i].copy() for i in top_indices],
        "top_k_fitness": [float(raw_fitness[i]) for i in top_indices],
        "config": {
            "pop_size": pop_size,
            "n_generations": n_generations,
            "tournament_k": tournament_k,
            "mutation_rate": mutation_rate,
            "sharing": sharing,
            "sharing_threshold": sharing_threshold,
            "top_k": top_k,
            "seed": seed,
            "rf_seed": rf_seed,
            "feature_cols": list(feature_cols),
        },
    }


def _init_population(
    pop_size: int, n_features: int, rng: np.random.Generator
) -> np.ndarray:
    pop = (rng.random((pop_size, n_features)) < 0.5).astype(np.int8)
    for i in range(pop_size):
        if pop[i].sum() == 0:
            pop[i, rng.integers(n_features)] = 1
    return pop


def _fitness(
    mask: np.ndarray,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    labels: list,
    rf_seed: int,
) -> float:
    if mask.sum() == 0:
        return 0.0
    cols = X_train.columns[mask.astype(bool)]
    rf = train_rf(X_train[cols], y_train, seed=rf_seed)
    y_pred = rf.predict(X_val[cols])
    return macro_f1(y_val, y_pred, labels=labels)


def _tournament(
    fitness: np.ndarray, k: int, rng: np.random.Generator
) -> int:
    contestants = rng.integers(0, len(fitness), size=k)
    return int(contestants[np.argmax(fitness[contestants])])


def uniform_crossover(
    p1: np.ndarray, p2: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    mask = rng.random(len(p1)) < 0.5
    child = np.where(mask, p1, p2)
    return child


def bit_flip_mutate(
    individual: np.ndarray, rate: float, rng: np.random.Generator
) -> np.ndarray:
    flips = rng.random(len(individual)) < rate
    return np.where(flips, 1 - individual, individual)


def apply_fitness_sharing(
    raw_fitness: np.ndarray,
    population: np.ndarray,
    threshold: float,
) -> np.ndarray:
    # Pairwise agreement: fraction of bits that match between two masks.
    diffs = population[:, None, :] != population[None, :, :]
    agreement = 1.0 - diffs.mean(axis=2)
    n_similar = (agreement >= threshold).sum(axis=1)
    return raw_fitness / n_similar


def _diversity(population: np.ndarray) -> float:
    pop_size = len(population)
    if pop_size < 2:
        return 0.0
    diffs = population[:, None, :] != population[None, :, :]
    pairwise = diffs.mean(axis=2)
    iu = np.triu_indices(pop_size, k=1)
    return float(pairwise[iu].mean())


def committee_predict(
    masks: list[np.ndarray],
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    train_split: str = "train",
    eval_split: str = "test",
    rf_seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    parts = split_features(features_df, feature_cols, label_col)
    X_train, y_train = parts[train_split]["X"], parts[train_split]["y"]
    X_eval, y_eval = parts[eval_split]["X"], parts[eval_split]["y"]

    proba_sum: np.ndarray | None = None
    classes_: np.ndarray | None = None
    used = 0
    for mask in masks:
        cols = [c for c, b in zip(feature_cols, mask) if b]
        if not cols:
            continue
        rf = train_rf(X_train[cols], y_train, seed=rf_seed)
        proba = rf.predict_proba(X_eval[cols])
        if proba_sum is None:
            proba_sum = proba
            classes_ = rf.classes_
        else:
            proba_sum = proba_sum + proba
        used += 1

    if used == 0:
        raise ValueError("All committee masks are empty")
    proba_avg = proba_sum / used
    y_pred = classes_[np.argmax(proba_avg, axis=1)]
    return y_eval.values, y_pred, proba_avg

Writing src/ec.py


In [17]:
%%writefile src/evaluate.py
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd

from src.baselines import (
    evaluate_predictions,
    split_features,
    train_rf,
)
from src.ec import committee_predict


def evaluate_mask_on_test(
    mask: np.ndarray,
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    method_name: str = "method",
    seed: int = 42,
) -> dict:
    cols = [c for c, b in zip(feature_cols, mask) if b]
    if not cols:
        raise ValueError("Mask selects zero features")
    parts = split_features(features_df, feature_cols, label_col)
    rf = train_rf(parts["train"]["X"][cols], parts["train"]["y"], seed=seed)
    y_pred = rf.predict(parts["test"]["X"][cols])
    labels = sorted(features_df[label_col].unique())
    metrics = evaluate_predictions(parts["test"]["y"], y_pred, labels)
    metrics["method"] = method_name
    metrics["selected_features"] = cols
    metrics["n_selected"] = len(cols)
    return metrics


def evaluate_committee_on_test(
    masks: list[np.ndarray],
    features_df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str = "primary_label",
    method_name: str = "ec_niched_committee",
    seed: int = 42,
) -> dict:
    y_true, y_pred, _ = committee_predict(
        masks,
        features_df,
        feature_cols,
        label_col=label_col,
        train_split="train",
        eval_split="test",
        rf_seed=seed,
    )
    labels = sorted(features_df[label_col].unique())
    metrics = evaluate_predictions(y_true, y_pred, labels)
    metrics["method"] = method_name
    metrics["committee_size"] = len(masks)
    metrics["selected_features_per_member"] = [
        [c for c, b in zip(feature_cols, m) if b] for m in masks
    ]
    return metrics


def comparison_table(method_results: list[dict]) -> pd.DataFrame:
    rows = []
    for m in method_results:
        row = {
            "method": m["method"],
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
        }
        for label, recall in m["per_species_recall"].items():
            row[f"recall_{label}"] = recall
        rows.append(row)
    return pd.DataFrame(rows).set_index("method")


def plot_comparison(comparison_df: pd.DataFrame, save_path: Path | None = None):
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    headline = comparison_df[["accuracy", "macro_f1"]]
    headline.plot(kind="bar", ax=axes[0], rot=30)
    axes[0].set_title("Overall accuracy and macro-F1 by method")
    axes[0].set_ylim([0, 1])
    axes[0].set_ylabel("score")
    axes[0].legend(loc="lower right")

    recall_cols = [c for c in comparison_df.columns if c.startswith("recall_")]
    per_species = comparison_df[recall_cols].copy()
    per_species.columns = [c.replace("recall_", "") for c in recall_cols]
    per_species.plot(kind="bar", rot=45, ax=axes[1])
    axes[1].set_title("Per-species recall by method")
    axes[1].set_ylim([0, 1])
    axes[1].set_ylabel("recall")
    axes[1].legend(loc="lower right", fontsize="small")

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig

Overwriting src/evaluate.py


## Stage 1. Data

We select 20 species from BirdCLEF+ 2026 with deliberately uneven clip counts to study performance under class imbalance. The slate spans three taxa, with counts ranging from 200 clips for the most common species (Rufous-bellied Thrush, Aves) down to 15 clips for the rarest (Lesser Snouted Tree Frog, Amphibia). Each clip is then split into training (70 percent), validation (15 percent), and held-out test (15 percent) sets. Splitting happens at the clip level rather than the window level so that windows from the same recording never appear in two splits.

In [11]:
from src import data

# Original 5 species for the baseline MLP experiment
SPECIES = [
    ("rubthr1", 200),   # Rufous-bellied Thrush (Aves)
    ("banana", 100),    # Bananaquit (Aves)
    ("244024", 50),     # Giant Cicada (Insecta)
    ("22973", 30),      # Whistling Grass Frog (Amphibia)
    ("24279", 15),      # Lesser Snouted Tree Frog (Amphibia)
]
SPLIT_RATIOS = (0.70, 0.15, 0.15)

manifest = data.build_manifest(
    raw_data_dir=RAW_DATA_DIR,
    species_specs=SPECIES,
    split_ratios=SPLIT_RATIOS,
    seed=MANIFEST_SEED,
)
manifest.to_csv(DATA_DIR / "manifest.csv", index=False)
print(f"Wrote {len(manifest)} clip rows to {DATA_DIR / 'manifest.csv'}")
data.split_counts(manifest)

Wrote 395 clip rows to /kaggle/working/data/manifest.csv


split,train,val,test
primary_label,,,
22973,21,4,5
24279,10,2,3
244024,35,8,7
banana,70,15,15
rubthr1,140,30,30


## Stage 2. Feature extraction

Each clip is segmented into non-overlapping 5-second windows. For each window we compute 34 numerical features using the librosa library, namely 13 Mel-frequency cepstral coefficients (MFCCs), spectral centroid, spectral rolloff, spectral flux, and zero-crossing rate, each summarized as mean and standard deviation across the window. The resulting table has one row per window and 34 feature columns plus species and split metadata. The table is cached to disk so subsequent stages do not need to re-extract.

In [12]:
from src import features

FEATURES_PATH = DATA_DIR / "features.pkl"

if FEATURES_PATH.exists():
    print(f"Loading cached features from {FEATURES_PATH}")
    features_df = pd.read_pickle(FEATURES_PATH)
else:
    print("Extracting features (slow first time)...")
    features_df = features.extract_all(manifest, raw_data_dir=RAW_DATA_DIR)
    features_df.to_pickle(FEATURES_PATH)
    print(f"Wrote {len(features_df)} window rows to {FEATURES_PATH}")

FEATURE_COLS = features.feature_columns()
print(f"{len(features_df)} windows  x  {len(FEATURE_COLS)} features")
features_df.head()

Extracting features (slow first time)...


Extracting features:   0%|          | 0/395 [00:00<?, ?it/s]

Wrote 2504 window rows to /kaggle/working/data/features.pkl
2504 windows  x  34 features


/kaggle/working/src/features.py:157: UserWarning: Skipped 26 clips (shorter than 5.0s or failed to load)
  warnings.warn(


,window_id,clip_id,primary_label,common_name,class_name,split,window_index,mfcc_0_mean,mfcc_1_mean,mfcc_2_mean,...,mfcc_11_std,mfcc_12_std,spectral_centroid_mean,spectral_centroid_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flux_mean,spectral_flux_std,zcr_mean,zcr_std
0,XC1003072_w0,XC1003072,rubthr1,Rufous-bellied Thrush,Aves,train,0,-307.627960,-1.387402,-76.860199,...,4.525420,5.652763,3688.591432,320.222263,6270.398966,899.799323,6.033416,10.663015,0.281614,0.038961
1,iNat1626515_w0,iNat1626515,rubthr1,Rufous-bellied Thrush,Aves,val,0,-475.494904,155.862366,-18.682686,...,9.477915,6.992568,1368.728256,255.150768,2425.824992,551.333318,1.515918,0.587699,0.084405,0.028353
2,XC587128_w0,XC587128,rubthr1,Rufous-bellied Thrush,Aves,train,0,-387.391724,124.761192,1.096541,...,9.329631,11.006266,1779.872611,799.899421,3595.895386,1745.191451,8.698405,7.545465,0.126881,0.077053
3,XC587128_w1,XC587128,rubthr1,Rufous-bellied Thrush,Aves,train,1,-234.899734,150.439178,16.659859,...,7.651182,10.554420,1253.357935,344.178418,2591.311646,1061.783446,26.753670,8.782391,0.062570,0.045997
4,XC587128_w2,XC587128,rubthr1,Rufous-bellied Thrush,Aves,train,2,-223.793488,150.748215,5.851923,...,10.778601,9.089325,1281.619528,373.654569,2638.066610,896.391841,23.370466,6.223459,0.066650,0.036418


## Stage 3. Baseline classifiers

Two reference classifiers establish the performance every later method must beat. The first is an **MLP** trained on all 34 features, with no selection. The second is an **MLP** trained on only the top 15 features picked by minimum Redundancy Maximum Relevance (mRMR), a well-known filter-based feature selection algorithm.

The actual baseline fits run inside the experiment loop in Stage 4, alongside the evolutionary search, so that each random seed produces its own baseline numbers for direct comparison.

## Stage 4. Evolutionary feature selection

The genetic algorithm searches over binary feature masks of length 34, where each bit decides whether to keep or drop one feature. An individual's fitness is the macro-averaged F1 score of an **MLP** classifier trained on the masked features and evaluated on the validation split. The search uses tournament selection (k=3), uniform crossover, and bit-flip mutation at rate 1/N, with a population of 50 individuals run for 50 generations.

We compare two modes of the GA. Plain EC applies no diversity preservation and serves as our ablation. Niched EC applies fitness sharing, dividing each individual's raw fitness by the number of others whose mask agrees on more than a threshold percentage of bits. This rewards individuals that explore underrepresented regions of the search space and prevents the population from converging on a single solution. The threshold is a hyperparameter we tune in the experiment loop.

For each random seed and niching threshold, we evaluate four EC methods on top of the two baselines. The single best individual from plain EC and from niched EC, and a committee of the top five individuals from each mode voting by averaged predicted probabilities. The niched committee is the proposal's primary contribution. The plain-EC committee is included as a control, since without it we cannot tell whether the gain comes from niching specifically or from the committee mechanism on its own. Per-method results are accumulated into a single `results.csv` across all (seed, threshold) combinations. The final top-K populations are persisted to `histories.pkl` so future voting-variant experiments can run off the saved artifact without rerunning the GA.

In [13]:
# Define MLP drop-in replacement for train_rf
# This must be defined BEFORE importing from src modules

import warnings

def train_mlp_as_rf(X_train, y_train, seed=42, **_unused):
    """Drop-in replacement for src.baselines.train_rf. Returns a fitted
    sklearn Pipeline (StandardScaler -> MLPClassifier) that implements
    fit / predict / predict_proba / classes_, matching the RF API the
    rest of the codebase expects."""
    mlp = MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN,
        solver=MLP_SOLVER,
        max_iter=MLP_MAX_ITER,
        random_state=seed,
    )
    pipe = Pipeline([('scale', StandardScaler()), ('mlp', mlp)])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')  # silence convergence warnings on small features
        pipe.fit(X_train, y_train)
    # Pipeline.classes_ is already a property that reads through to the final
    # estimator, so committee_predict's rf.classes_ access works out of the box.
    return pipe

print(f"train_mlp_as_rf defined with MLP{MLP_HIDDEN}, solver={MLP_SOLVER}, max_iter={MLP_MAX_ITER}")

train_mlp_as_rf defined with MLP(16,), solver=adam, max_iter=200


In [18]:
from src import ec, baselines, evaluate

POP_SIZE = 50
N_GENERATIONS = 50
TOURNAMENT_K = 3
TOP_K_COMMITTEE = 5
K_MRMR = 15

# For a single-seed run, keep these as one-element lists.
# For multi-seed:    SEEDS = [42, 43, 44, 45, 46]
# For threshold sweep: THRESHOLDS = [0.9, 0.95]
SEEDS = [42, 43, 44, 45, 46]
THRESHOLDS = [0.9, 0.95]

In [ ]:
import datetime
import json
import pickle

# Monkey-patch the train_rf binding in every module that uses it. This is
# safe because every train_rf call site treats its return value as an
# opaque classifier with fit/predict/predict_proba/classes_.
baselines.train_rf = train_mlp_as_rf
ec.train_rf = train_mlp_as_rf
evaluate.train_rf = train_mlp_as_rf

print(f"Inner classifier swapped to MLPClassifier{MLP_HIDDEN} solver={MLP_SOLVER} max_iter={MLP_MAX_ITER}")

# Auto-generate RUN_NAME unless already defined for a named experiment.
RUN_ENV = "kaggle_mlp"
try:
    RUN_NAME
except NameError:
    n_runs = len(SEEDS) * len(THRESHOLDS)
    RUN_NAME = (
        f"{RUN_ENV}_n{n_runs}_seeds{len(SEEDS)}_thresh{len(THRESHOLDS)}"
        f"_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
RUN_DIR = RESULTS_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run name: {RUN_NAME}")
print(f"Run dir : {RUN_DIR}")

config = {
    "run_name": RUN_NAME,
    "env": RUN_ENV,
    "seeds": SEEDS,
    "thresholds": THRESHOLDS,
    "pop_size": POP_SIZE,
    "n_generations": N_GENERATIONS,
    "tournament_k": TOURNAMENT_K,
    "top_k_committee": TOP_K_COMMITTEE,
    "k_mrmr": K_MRMR,
    "manifest_seed": MANIFEST_SEED,
    "species": [{"label": s, "count": n} for s, n in SPECIES],
    "split_ratios": list(SPLIT_RATIOS),
    "inner_classifier": "MLPClassifier",
    "mlp_hidden": list(MLP_HIDDEN),
    "mlp_solver": MLP_SOLVER,
    "mlp_max_iter": MLP_MAX_ITER,
}
with open(RUN_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

all_rows = []
all_histories = {}

for seed in SEEDS:
    print(f"=== Seed {seed} ===")

    print("  baselines...")
    af = baselines.run_baseline_all_features(features_df, FEATURE_COLS, seed=seed)
    mr = baselines.run_baseline_mrmr(features_df, FEATURE_COLS, k=K_MRMR, seed=seed)

    print("  plain EC...")
    plain = ec.run_ga(
        features_df, FEATURE_COLS,
        pop_size=POP_SIZE, n_generations=N_GENERATIONS,
        tournament_k=TOURNAMENT_K, sharing=False,
        top_k=TOP_K_COMMITTEE, seed=seed, rf_seed=seed,
        progress=True,
    )
    plain_eval = evaluate.evaluate_mask_on_test(
        plain["best_mask"], features_df, FEATURE_COLS,
        method_name="ec_plain_best", seed=seed,
    )
    plain_committee_eval = evaluate.evaluate_committee_on_test(
        plain["top_k_masks"], features_df, FEATURE_COLS,
        method_name="ec_plain_committee", seed=seed,
    )
    all_histories[(seed, None, "plain")] = {
        "history": plain["history"],
        "final_population": plain["final_population"].tolist(),
        "raw_fitness": plain["raw_fitness"].tolist(),
        "top_k_masks": [m.tolist() for m in plain["top_k_masks"]],
        "top_k_fitness": plain["top_k_fitness"],
        "best_mask": plain["best_mask"].tolist(),
        "best_fitness": plain["best_fitness"],
    }

    for m in (af, mr, plain_eval, plain_committee_eval):
        all_rows.append({
            "seed": seed,
            "threshold": None,
            "method": m["method"],
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            **{f"recall_{label}": v for label, v in m["per_species_recall"].items()},
        })

    for thresh in THRESHOLDS:
        print(f"  niched EC, threshold={thresh}...")
        niched = ec.run_ga(
            features_df, FEATURE_COLS,
            pop_size=POP_SIZE, n_generations=N_GENERATIONS,
            tournament_k=TOURNAMENT_K, sharing=True,
            sharing_threshold=thresh,
            top_k=TOP_K_COMMITTEE, seed=seed, rf_seed=seed,
            progress=True,
        )
        niched_eval = evaluate.evaluate_mask_on_test(
            niched["best_mask"], features_df, FEATURE_COLS,
            method_name="ec_niched_best", seed=seed,
        )
        committee_eval = evaluate.evaluate_committee_on_test(
            niched["top_k_masks"], features_df, FEATURE_COLS,
            method_name="ec_niched_committee", seed=seed,
        )
        all_histories[(seed, thresh, "niched")] = {
            "history": niched["history"],
            "final_population": niched["final_population"].tolist(),
            "raw_fitness": niched["raw_fitness"].tolist(),
            "top_k_masks": [m.tolist() for m in niched["top_k_masks"]],
            "top_k_fitness": niched["top_k_fitness"],
            "best_mask": niched["best_mask"].tolist(),
            "best_fitness": niched["best_fitness"],
        }

        for m in (niched_eval, committee_eval):
            all_rows.append({
                "seed": seed,
                "threshold": thresh,
                "method": m["method"],
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                **{f"recall_{label}": v for label, v in m["per_species_recall"].items()},
            })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(RUN_DIR / "results.csv", index=False)
print(f"Wrote {len(results_df)} rows to {RUN_DIR / 'results.csv'}")

with open(RUN_DIR / "histories.pkl", "wb") as f:
    pickle.dump(all_histories, f)

results_df.round(3)

Inner classifier swapped to MLPClassifier(16,) solver=adam max_iter=200
Run name: kaggle_mlp_n10_seeds5_thresh2_20260612_025331
Run dir : /kaggle/working/results/runs/kaggle_mlp_n10_seeds5_thresh2_20260612_025331
=== Seed 42 ===
  baselines...
  plain EC...


GA generations:   0%|          | 0/50 [00:00<?, ?it/s]

  niched EC, threshold=0.9...


GA generations:   0%|          | 0/50 [00:00<?, ?it/s]

  niched EC, threshold=0.95...


...

GA generations:   0%|          | 0/50 [00:00<?, ?it/s]

=== Seed 45 ===
  baselines...
  plain EC...


GA generations:   0%|          | 0/50 [00:00<?, ?it/s]

  niched EC, threshold=0.9...


GA generations:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (seed, thresh, mode), entry in all_histories.items():
    h = pd.DataFrame(entry["history"])
    label = f"s{seed}, {mode}" + (f", t={thresh}" if thresh is not None else "")
    axes[0].plot(h["generation"], h["best_raw"], label=label, alpha=0.7)
    axes[1].plot(h["generation"], h["diversity"], label=label, alpha=0.7)
axes[0].set_title("Best raw fitness per generation")
axes[0].set_xlabel("generation")
axes[0].set_ylabel("macro-F1 (validation)")
axes[0].legend(fontsize="small", loc="lower right")
axes[1].set_title("Population diversity (mean pairwise Hamming)")
axes[1].set_xlabel("generation")
axes[1].set_ylabel("diversity")
axes[1].legend(fontsize="small", loc="lower right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ec_history.png", dpi=150, bbox_inches="tight")
plt.show()

## Stage 5. Evaluation

The experiment loop above produces one row per (random seed, niching threshold, method) combination on the held-out test set. Six methods are compared, two baselines (all features and mRMR top-15), two plain-EC variants (the single best individual and the top-5 committee, both serving as ablations against fitness sharing), and two niched variants (niched EC's best individual and the niched EC committee). The metrics reported are overall accuracy, macro-averaged F1 (which weights every species equally regardless of frequency), and per-species recall.

When multiple seeds are run, results are aggregated as mean and standard deviation across seeds. The accompanying plot shows accuracy, macro-F1, and per-species recall per method.

In [ ]:
metric_cols = ["accuracy", "macro_f1"] + [c for c in results_df.columns if c.startswith("recall_")]

if len(SEEDS) > 1:
    summary = results_df.groupby("method")[metric_cols].agg(["mean", "std"]).round(3)
    print(f"Summary across {len(SEEDS)} seeds (MLP inner classifier)")
else:
    summary = results_df.set_index("method")[metric_cols].round(3)
    print(f"Single seed (seed={SEEDS[0]}) with MLP inner classifier")
summary

In [ ]:
mean_per_method = results_df.groupby("method")[metric_cols].mean()
fig = evaluate.plot_comparison(
    mean_per_method, save_path=FIGURES_DIR / "comparison.png"
)
plt.show()

## Cross-run comparison

If multiple experiments have been run by varying the SEEDS or THRESHOLDS lists in Stage 4, the cell below stacks every saved run under `results/runs/` into a single DataFrame for comparison. Useful when sweeping configurations.

In [ ]:
import json
from pathlib import Path

import pandas as pd

runs_root = RESULTS_DIR / "runs"
if not runs_root.exists():
    print("No results/runs/ directory yet.")
else:
    all_dfs = []
    for run_dir in sorted(runs_root.iterdir()):
        if not run_dir.is_dir():
            continue
        # New format (multi-seed): results.csv. Old format: comparison.csv.
        for fname in ("results.csv", "comparison.csv"):
            comp_path = run_dir / fname
            if comp_path.exists():
                break
        else:
            continue
        df = pd.read_csv(comp_path)
        df["run_name"] = run_dir.name
        cfg_path = run_dir / "config.json"
        if cfg_path.exists():
            with open(cfg_path) as f:
                cfg = json.load(f)
            df["env"] = cfg.get("env", "unknown")
        all_dfs.append(df)
    if all_dfs:
        all_runs = pd.concat(all_dfs, ignore_index=True)
        head_cols = ["run_name", "env", "seed", "threshold", "method", "accuracy", "macro_f1"]
        recall_cols = [c for c in all_runs.columns if c.startswith("recall_")]
        cols = [c for c in head_cols + recall_cols if c in all_runs.columns]
        print(f"Aggregated {len(all_runs)} method-rows from {len(all_dfs)} runs")
        all_runs[cols].round(3)
    else:
        print("No runs in results/runs/ yet.")

## Additional Visualizations

This section provides supplementary visualizations for better understanding of the data distribution, feature extraction, and model behavior.

### Original Data Distribution

Visualize the distribution of species in the dataset, including class imbalance and train/val/test splits.

In [ ]:
# Data Distribution Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Species distribution by split
split_counts = manifest.groupby(['primary_label', 'split']).size().unstack(fill_value=0)
split_counts.plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[0, 0].set_title('Species Distribution by Split', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Species Label')
axes[0, 0].set_ylabel('Number of Clips')
axes[0, 0].legend(title='Split')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Class imbalance visualization
species_counts = manifest['primary_label'].value_counts()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(species_counts)))
axes[0, 1].bar(range(len(species_counts)), species_counts.values, color=colors)
axes[0, 1].set_title('Class Imbalance Pattern', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Species (ranked by frequency)')
axes[0, 1].set_ylabel('Number of Clips')
axes[0, 1].set_xticks(range(len(species_counts)))
axes[0, 1].set_xticklabels(species_counts.index, rotation=45, ha='right')

# Add count labels on bars
for i, v in enumerate(species_counts.values):
    axes[0, 1].text(i, v + 2, str(v), ha='center', fontsize=9)

# 3. Taxonomic distribution (class_name)
taxon_counts = manifest['class_name'].value_counts()
axes[1, 0].pie(taxon_counts.values, labels=taxon_counts.index, autopct='%1.1f%%',
               colors=['#3498db', '#2ecc71', '#f39c12'], startangle=90)
axes[1, 0].set_title('Distribution by Taxonomic Class', fontsize=12, fontweight='bold')

# 4. Split proportions
split_totals = manifest['split'].value_counts()
axes[1, 1].pie(split_totals.values, labels=split_totals.index, autopct='%1.1f%%',
               colors=['#2ecc71', '#3498db', '#e74c3c'], startangle=90,
               explode=(0.02, 0.02, 0.02))
axes[1, 1].set_title('Overall Split Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDataset Summary:")
print(f"  Total clips: {len(manifest)}")
print(f"  Species: {manifest['primary_label'].nunique()}")
print(f"  Taxa: {manifest['class_name'].nunique()} (Aves, Insecta, Amphibia)")
print(f"\nImbalance ratio (most common : rarest): {species_counts.iloc[0]} : {species_counts.iloc[-1]}")

### Feature Extraction Visualization

Visualize the extracted audio features: distributions, correlations, and feature importance patterns.

In [ ]:
# Feature Distribution Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature count by category
mfcc_features = [c for c in FEATURE_COLS if c.startswith('mfcc_')]
spectral_features = [c for c in FEATURE_COLS if any(x in c for x in ['spectral', 'rolloff', 'flux'])]
zcr_features = [c for c in FEATURE_COLS if 'zcr' in c]

feature_categories = {
    'MFCC': len(mfcc_features),
    'Spectral': len(spectral_features),
    'Zero-Crossing': len(zcr_features)
}

axes[0, 0].bar(feature_categories.keys(), feature_categories.values(),
               color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Features by Category', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Number of Features')
for i, (k, v) in enumerate(feature_categories.items()):
    axes[0, 0].text(i, v + 0.5, str(v), ha='center', fontsize=10)

# 2. Feature value distributions (sample a few key features)
key_features = ['mfcc_0_mean', 'spectral_centroid_mean', 'zcr_mean']
for i, feat in enumerate(key_features):
    if feat in features_df.columns:
        row, col = (0, 1) if i == 0 else (1, 0) if i == 1 else (1, 1)
        for species in features_df['primary_label'].unique():
            subset = features_df[features_df['primary_label'] == species][feat]
            axes[row, col].hist(subset, alpha=0.5, label=species, bins=20)
        axes[row, col].set_title(f'Distribution of {feat}', fontsize=10, fontweight='bold')
        axes[row, col].set_xlabel('Value')
        axes[row, col].set_ylabel('Frequency')
        axes[row, col].legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Correlation Heatmap
plt.figure(figsize=(12, 10))

# Select a subset of features for readability (mean features only)
mean_features = [c for c in FEATURE_COLS if '_mean' in c]
corr_matrix = features_df[mean_features].corr()

import seaborn as sns
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix (Mean Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Statistics by Species
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Mean MFCC values by species
mfcc_mean_cols = [c for c in FEATURE_COLS if c.startswith('mfcc_') and c.endswith('_mean')]
if mfcc_mean_cols:
    mfcc_by_species = features_df.groupby('primary_label')[mfcc_mean_cols].mean()
    im = axes[0].imshow(mfcc_by_species.values, aspect='auto', cmap='viridis')
    axes[0].set_xticks(range(len(mfcc_mean_cols)))
    axes[0].set_xticklabels([c.replace('mfcc_', '').replace('_mean', '') for c in mfcc_mean_cols],
                            rotation=45, ha='right')
    axes[0].set_yticks(range(len(mfcc_by_species.index)))
    axes[0].set_yticklabels(mfcc_by_species.index)
    axes[0].set_title('Mean MFCC Coefficients by Species', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('MFCC Coefficient')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# 2. Spectral feature comparison
spectral_cols = ['spectral_centroid_mean', 'spectral_rolloff_mean', 'spectral_flux_mean']
if all(c in features_df.columns for c in spectral_cols):
    spectral_by_species = features_df.groupby('primary_label')[spectral_cols].mean()
    spectral_by_species.plot(kind='bar', ax=axes[1], width=0.8)
    axes[1].set_title('Spectral Features by Species', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Species')
    axes[1].set_ylabel('Mean Value')
    axes[1].legend(loc='best', fontsize=8)
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_by_species.png', dpi=150, bbox_inches='tight')
plt.show()

### Evolutionary Search Visualization

Detailed analysis of the genetic algorithm behavior and fitness landscape.

In [ ]:
# GA Convergence Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for (seed, thresh, mode), entry in all_histories.items():
    h = pd.DataFrame(entry['history'])
    label = f"s{seed}, {mode}" + (f", t={thresh}" if thresh is not None else "")
    
    # Raw fitness evolution
    axes[0, 0].plot(h['generation'], h['best_raw'], label=label, alpha=0.7, linewidth=2)
    axes[0, 0].fill_between(h['generation'], h['mean_raw'] - h['mean_raw'].std(),
                            h['mean_raw'] + h['mean_raw'].std(), alpha=0.1)
    
    # Shared fitness (if available)
    if 'best_shared' in h.columns:
        axes[0, 1].plot(h['generation'], h['best_shared'], label=label, alpha=0.7, linewidth=2)
    
    # Diversity over time
    axes[1, 0].plot(h['generation'], h['diversity'], label=label, alpha=0.7, linewidth=2)
    
    # Fitness vs Diversity scatter (final generation)
    axes[1, 1].scatter(h['diversity'].iloc[-1], h['best_raw'].iloc[-1],
                       s=100, alpha=0.6, label=label)

axes[0, 0].set_title('Best Raw Fitness over Generations', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Generation')
axes[0, 0].set_ylabel('Macro-F1 Score')
axes[0, 0].legend(fontsize=8, loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_title('Best Shared Fitness over Generations', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Generation')
axes[0, 1].set_ylabel('Shared Fitness')
axes[0, 1].legend(fontsize=8, loc='lower right')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].set_title('Population Diversity over Generations', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Generation')
axes[1, 0].set_ylabel('Mean Pairwise Hamming Distance')
axes[1, 0].legend(fontsize=8, loc='best')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_title('Final Fitness vs Diversity', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Diversity')
axes[1, 1].set_ylabel('Best Raw Fitness')
axes[1, 1].legend(fontsize=8, loc='best')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'ga_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Selection Patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract top-k masks from final generation
for (seed, thresh, mode), entry in all_histories.items():
    if 'top_k_masks' in entry:
        masks = np.array(entry['top_k_masks'])
        feature_usage = masks.mean(axis=0)  # Fraction of committee using each feature
        
        label = f"{mode}, s={seed}" + (f", t={thresh}" if thresh is not None else "")
        axes[0].barh(range(len(feature_usage)), feature_usage, alpha=0.6, label=label)
        
        # Feature frequency distribution
        axes[1].hist(feature_usage, bins=20, alpha=0.5, label=label)

axes[0].set_yticks(range(len(FEATURE_COLS)))
axes[0].set_yticklabels(FEATURE_COLS, fontsize=7)
axes[0].set_xlabel('Selection Frequency (Top-K Committee)')
axes[0].set_title('Feature Usage in Top-K Committee', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')

axes[1].set_xlabel('Selection Frequency')
axes[1].set_ylabel('Number of Features')
axes[1].set_title('Distribution of Feature Selection Frequencies', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_selection_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# DETAILED FEATURE SELECTION ANALYSIS
# ==========================================

print("\n" + "="*70)
print("FEATURE SELECTION ANALYSIS")
print("="*70)

# Aggregate feature selection across all runs
feature_selection_stats = {}

for feature in FEATURE_COLS:
    feature_selection_stats[feature] = {
        'total_selections': 0,
        'total_runs': 0,
        'selection_frequency': 0.0,
        'by_method': {}
    }

# Analyze selections from all_histories
for (seed, thresh, mode), entry in all_histories.items():
    if 'top_k_masks' in entry:
        masks = np.array(entry['top_k_masks'])  # Shape: (committee_size, n_features)
        method_name = f"{mode}_thresh{thresh}" if thresh else mode
        
        for i, feature in enumerate(FEATURE_COLS):
            # Count how many committee members selected this feature
            selections = masks[:, i].sum()
            feature_selection_stats[feature]['total_selections'] += int(selections)
            feature_selection_stats[feature]['total_runs'] += len(masks)
            
            if method_name not in feature_selection_stats[feature]['by_method']:
                feature_selection_stats[feature]['by_method'][method_name] = []
            feature_selection_stats[feature]['by_method'][method_name].append(float(selections / len(masks)))

# Calculate overall selection frequencies
for feature in FEATURE_COLS:
    stats = feature_selection_stats[feature]
    if stats['total_runs'] > 0:
        stats['selection_frequency'] = stats['total_selections'] / stats['total_runs']

# Sort features by selection frequency
sorted_features = sorted(
    feature_selection_stats.items(),
    key=lambda x: x[1]['selection_frequency'],
    reverse=True
)

# Display results
print("\n📊 TOP 10 MOST SELECTED FEATURES:")
print("-"*50)
for i, (feature, stats) in enumerate(sorted_features[:10]):
    freq = stats['selection_frequency'] * 100
    print(f"  {i+1:2d}. {feature:30s} {freq:5.1f}% ({stats['total_selections']}/{stats['total_runs']})")

print("\n📊 LEAST SELECTED FEATURES (Bottom 5):")
print("-"*50)
for i, (feature, stats) in enumerate(sorted_features[-5:]):
    freq = stats['selection_frequency'] * 100
    print(f"  {i+1:2d}. {feature:30s} {freq:5.1f}% ({stats['total_selections']}/{stats['total_runs']})")

# Create comprehensive feature selection visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Feature selection frequency bar chart
features = [f[0] for f in sorted_features]
frequencies = [f[1]['selection_frequency'] * 100 for f in sorted_features]
colors = ['#2ecc71' if f > 50 else '#f39c12' if f > 25 else '#e74c3c' for f in frequencies]

axes[0, 0].barh(range(len(features)), frequencies, color=colors)
axes[0, 0].set_yticks(range(len(features)))
axes[0, 0].set_yticklabels(features, fontsize=8)
axes[0, 0].set_xlabel('Selection Frequency (%)')
axes[0, 0].set_title('Feature Selection Frequency (All Runs)', fontsize=11, fontweight='bold')
axes[0, 0].axvline(x=50, color='green', linestyle='--', alpha=0.5, label='50% threshold')
axes[0, 0].axvline(x=25, color='orange', linestyle='--', alpha=0.5, label='25% threshold')
axes[0, 0].legend()

# 2. Selection frequency distribution
axes[0, 1].hist(frequencies, bins=15, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Selection Frequency (%)')
axes[0, 1].set_ylabel('Number of Features')
axes[0, 1].set_title('Distribution of Selection Frequencies', fontsize=11, fontweight='bold')
axes[0, 1].axvline(x=50, color='green', linestyle='--', alpha=0.7, label='Median (highly selected)')
median_freq = np.median(frequencies)
axes[0, 1].axvline(x=median_freq, color='red', linestyle='-', alpha=0.7, label=f'Actual median: {median_freq:.1f}%')
axes[0, 1].legend()

# 3. Feature category analysis
feature_categories = {
    'MFCC': [f for f in FEATURE_COLS if 'mfcc' in f],
    'Spectral': [f for f in FEATURE_COLS if 'spectral' in f],
    'Temporal': [f for f in FEATURE_COLS if any(x in f for x in ['zero_crossing', 'rms'])]
}

category_avg_freq = {}
for cat, cat_features in feature_categories.items():
    cat_freqs = [feature_selection_stats[f]['selection_frequency'] * 100 for f in cat_features if f in feature_selection_stats]
    category_avg_freq[cat] = np.mean(cat_freqs) if cat_freqs else 0

axes[1, 0].bar(category_avg_freq.keys(), category_avg_freq.values(),
               color=['#9b59b6', '#3498db', '#e74c3c'], edgecolor='black')
axes[1, 0].set_ylabel('Average Selection Frequency (%)')
axes[1, 0].set_title('Average Selection by Feature Category', fontsize=11, fontweight='bold')
for i, (cat, freq) in enumerate(category_avg_freq.items()):
    axes[1, 0].text(i, freq + 1, f'{freq:.1f}%', ha='center', fontweight='bold')

# 4. Compare niched vs plain selection patterns
niched_freqs = []
plain_freqs = []

for feature in FEATURE_COLS:
    niched_sum = 0
    niched_count = 0
    plain_sum = 0
    plain_count = 0
    
    for method, freqs in feature_selection_stats[feature]['by_method'].items():
        if 'niched' in method:
            niched_sum += sum(freqs)
            niched_count += len(freqs)
        elif 'plain' in method:
            plain_sum += sum(freqs)
            plain_count += len(freqs)
    
    if niched_count > 0:
        niched_freqs.append(niched_sum / niched_count * 100)
    if plain_count > 0:
        plain_freqs.append(plain_sum / plain_count * 100)

x = np.arange(len(FEATURE_COLS))
width = 0.35
axes[1, 1].bar(x - width/2, niched_freqs, width, label='Niched EC', color='#2ecc71', alpha=0.8)
axes[1, 1].bar(x + width/2, plain_freqs, width, label='Plain EC', color='#e74c3c', alpha=0.8)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_ylabel('Selection Frequency (%)')
axes[1, 1].set_title('Niched vs Plain EC Feature Selection', fontsize=11, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'detailed_feature_selection_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Save feature selection statistics to CSV
stats_df = pd.DataFrame([
    {
        'feature': feature,
        'selection_frequency': stats['selection_frequency'],
        'total_selections': stats['total_selections'],
        'total_runs': stats['total_runs'],
        'category': 'MFCC' if 'mfcc' in feature else 'Spectral' if 'spectral' in feature else 'Temporal'
    }
    for feature, stats in sorted_features
])
stats_df.to_csv(RUN_DIR / 'feature_selection_statistics.csv', index=False)
print(f"\n💾 Feature selection statistics saved to: {RUN_DIR / 'feature_selection_statistics.csv'}")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"  Total features: {len(FEATURE_COLS)}")
print(f"  Features selected >50% of time: {sum(1 for f in frequencies if f > 50)}")
print(f"  Features selected >25% of time: {sum(1 for f in frequencies if f > 25)}")
print(f"  Features selected <10% of time: {sum(1 for f in frequencies if f < 10)}")
print(f"  Average selection frequency: {np.mean(frequencies):.1f}%")
print(f"  Median selection frequency: {np.median(frequencies):.1f}%")


### Model Performance Analysis

Per-species performance breakdown and confusion analysis.

In [ ]:
# Per-Species Performance Heatmap
if 'results_df' in locals() and len(results_df) > 0:
    recall_cols = [c for c in results_df.columns if c.startswith('recall_')]
    if recall_cols:
        plt.figure(figsize=(14, 8))
        
        # Pivot data for heatmap
        heatmap_data = results_df.groupby('method')[recall_cols].mean()
        heatmap_data.columns = [c.replace('recall_', '') for c in heatmap_data.columns]
        
        sns.heatmap(heatmap_data, annot=True, cmap='RdYlGn', vmin=0, vmax=1,
                    fmt='.3f', linewidths=0.5, cbar_kws={'label': 'Recall'})
        plt.title('Per-Species Recall by Method', fontsize=14, fontweight='bold')
        plt.xlabel('Species Label')
        plt.ylabel('Method')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'per_species_recall_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # Print best method per species
        print("\nBest Method per Species:")
        for species in heatmap_data.columns:
            best_method = heatmap_data[species].idxmax()
            best_score = heatmap_data[species].max()
            print(f"  {species}: {best_method} ({best_score:.3f})")

In [ ]:
# Accuracy vs Macro-F1 Trade-off
if 'results_df' in locals() and len(results_df) > 0:
    plt.figure(figsize=(10, 7))
    
    # Group by method and calculate means
    method_stats = results_df.groupby('method')[['accuracy', 'macro_f1']].agg(['mean', 'std'])
    
    for method in method_stats.index:
        x = method_stats.loc[method, ('accuracy', 'mean')]
        y = method_stats.loc[method, ('macro_f1', 'mean')]
        xerr = method_stats.loc[method, ('accuracy', 'std')]
        yerr = method_stats.loc[method, ('macro_f1', 'std')]
        
        plt.errorbar(x, y, xerr=xerr, yerr=yerr, marker='o', markersize=10,
                     capsize=5, label=method, alpha=0.7)
    
    plt.xlabel('Accuracy', fontsize=12)
    plt.ylabel('Macro-F1', fontsize=12)
    plt.title('Accuracy vs Macro-F1 Trade-off by Method', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    
    # Add diagonal reference line (balanced performance)
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='balanced')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'accuracy_vs_f1_tradeoff.png', dpi=150, bbox_inches='tight')
    plt.show()

### Summary Statistics

Final summary of all visualizations and key metrics.

In [ ]:
# Generate Summary Report
print("=" * 60)
print("VISUALIZATION SUMMARY")
print("=" * 60)

print("\n📊 Data Distribution:")
print(f"  • Total clips: {len(manifest)}")
print(f"  • Windows extracted: {len(features_df)}")
print(f"  • Features per window: {len(FEATURE_COLS)}")
print(f"  • Species: {manifest['primary_label'].unique().tolist()}")

print("\n🔧 MLP Configuration:")
print(f"  • Hidden layers: {MLP_HIDDEN}")
print(f"  • Solver: {MLP_SOLVER}")
print(f"  • Max iterations: {MLP_MAX_ITER}")

print("\n🧬 Evolutionary Search:")
print(f"  • Population size: {POP_SIZE}")
print(f"  • Generations: {N_GENERATIONS}")
print(f"  • Seeds run: {SEEDS}")
print(f"  • Niching thresholds: {THRESHOLDS}")

print("\n📁 Saved Figures:")
for fig_path in sorted(FIGURES_DIR.glob('*.png')):
    print(f"  • {fig_path.name}")

print("\n" + "=" * 60)

In [ ]:
# ==========================================
# DETAILED FEATURE SELECTION ANALYSIS
# ==========================================

print("\n" + "="*70)
print("FEATURE SELECTION ANALYSIS")
print("="*70)

# Aggregate feature selection across all runs
feature_selection_stats = {}

for feature in FEATURE_COLS:
    feature_selection_stats[feature] = {
        'total_selections': 0,
        'total_runs': 0,
        'selection_frequency': 0.0,
        'by_method': {}
    }

# Analyze selections from all_histories
for (seed, thresh, mode), entry in all_histories.items():
    if 'top_k_masks' in entry:
        masks = np.array(entry['top_k_masks'])  # Shape: (committee_size, n_features)
        method_name = f"{mode}_thresh{thresh}" if thresh else mode
        
        for i, feature in enumerate(FEATURE_COLS):
            # Count how many committee members selected this feature
            selections = masks[:, i].sum()
            feature_selection_stats[feature]['total_selections'] += int(selections)
            feature_selection_stats[feature]['total_runs'] += len(masks)
            
            if method_name not in feature_selection_stats[feature]['by_method']:
                feature_selection_stats[feature]['by_method'][method_name] = []
            feature_selection_stats[feature]['by_method'][method_name].append(float(selections / len(masks)))

# Calculate overall selection frequencies
for feature in FEATURE_COLS:
    stats = feature_selection_stats[feature]
    if stats['total_runs'] > 0:
        stats['selection_frequency'] = stats['total_selections'] / stats['total_runs']

# Sort features by selection frequency
sorted_features = sorted(
    feature_selection_stats.items(),
    key=lambda x: x[1]['selection_frequency'],
    reverse=True
)

# Display results
print("\n📊 TOP 10 MOST SELECTED FEATURES:")
print("-"*50)
for i, (feature, stats) in enumerate(sorted_features[:10]):
    freq = stats['selection_frequency'] * 100
    print(f"  {i+1:2d}. {feature:30s} {freq:5.1f}% ({stats['total_selections']}/{stats['total_runs']})")

print("\n📊 LEAST SELECTED FEATURES (Bottom 5):")
print("-"*50)
for i, (feature, stats) in enumerate(sorted_features[-5:]):
    freq = stats['selection_frequency'] * 100
    print(f"  {i+1:2d}. {feature:30s} {freq:5.1f}% ({stats['total_selections']}/{stats['total_runs']})")

# Create comprehensive feature selection visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Feature selection frequency bar chart
features = [f[0] for f in sorted_features]
frequencies = [f[1]['selection_frequency'] * 100 for f in sorted_features]
colors = ['#2ecc71' if f > 50 else '#f39c12' if f > 25 else '#e74c3c' for f in frequencies]

axes[0, 0].barh(range(len(features)), frequencies, color=colors)
axes[0, 0].set_yticks(range(len(features)))
axes[0, 0].set_yticklabels(features, fontsize=8)
axes[0, 0].set_xlabel('Selection Frequency (%)')
axes[0, 0].set_title('Feature Selection Frequency (All Runs)', fontsize=11, fontweight='bold')
axes[0, 0].axvline(x=50, color='green', linestyle='--', alpha=0.5, label='50% threshold')
axes[0, 0].axvline(x=25, color='orange', linestyle='--', alpha=0.5, label='25% threshold')
axes[0, 0].legend()

# 2. Selection frequency distribution
axes[0, 1].hist(frequencies, bins=15, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Selection Frequency (%)')
axes[0, 1].set_ylabel('Number of Features')
axes[0, 1].set_title('Distribution of Selection Frequencies', fontsize=11, fontweight='bold')
axes[0, 1].axvline(x=50, color='green', linestyle='--', alpha=0.7, label='Median (highly selected)')
median_freq = np.median(frequencies)
axes[0, 1].axvline(x=median_freq, color='red', linestyle='-', alpha=0.7, label=f'Actual median: {median_freq:.1f}%')
axes[0, 1].legend()

# 3. Feature category analysis
feature_categories = {
    'MFCC': [f for f in FEATURE_COLS if 'mfcc' in f],
    'Spectral': [f for f in FEATURE_COLS if 'spectral' in f],
    'Temporal': [f for f in FEATURE_COLS if any(x in f for x in ['zero_crossing', 'rms'])]
}

category_avg_freq = {}
for cat, cat_features in feature_categories.items():
    cat_freqs = [feature_selection_stats[f]['selection_frequency'] * 100 for f in cat_features if f in feature_selection_stats]
    category_avg_freq[cat] = np.mean(cat_freqs) if cat_freqs else 0

axes[1, 0].bar(category_avg_freq.keys(), category_avg_freq.values(),
               color=['#9b59b6', '#3498db', '#e74c3c'], edgecolor='black')
axes[1, 0].set_ylabel('Average Selection Frequency (%)')
axes[1, 0].set_title('Average Selection by Feature Category', fontsize=11, fontweight='bold')
for i, (cat, freq) in enumerate(category_avg_freq.items()):
    axes[1, 0].text(i, freq + 1, f'{freq:.1f}%', ha='center', fontweight='bold')

# 4. Compare niched vs plain selection patterns
niched_freqs = []
plain_freqs = []

for feature in FEATURE_COLS:
    niched_sum = 0
    niched_count = 0
    plain_sum = 0
    plain_count = 0
    
    for method, freqs in feature_selection_stats[feature]['by_method'].items():
        if 'niched' in method:
            niched_sum += sum(freqs)
            niched_count += len(freqs)
        elif 'plain' in method:
            plain_sum += sum(freqs)
            plain_count += len(freqs)
    
    if niched_count > 0:
        niched_freqs.append(niched_sum / niched_count * 100)
    if plain_count > 0:
        plain_freqs.append(plain_sum / plain_count * 100)

x = np.arange(len(FEATURE_COLS))
width = 0.35
axes[1, 1].bar(x - width/2, niched_freqs, width, label='Niched EC', color='#2ecc71', alpha=0.8)
axes[1, 1].bar(x + width/2, plain_freqs, width, label='Plain EC', color='#e74c3c', alpha=0.8)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_ylabel('Selection Frequency (%)')
axes[1, 1].set_title('Niched vs Plain EC Feature Selection', fontsize=11, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'detailed_feature_selection_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Save feature selection statistics to CSV
stats_df = pd.DataFrame([
    {
        'feature': feature,
        'selection_frequency': stats['selection_frequency'],
        'total_selections': stats['total_selections'],
        'total_runs': stats['total_runs'],
        'category': 'MFCC' if 'mfcc' in feature else 'Spectral' if 'spectral' in feature else 'Temporal'
    }
    for feature, stats in sorted_features
])
stats_df.to_csv(RUN_DIR / 'feature_selection_statistics.csv', index=False)
print(f"\n💾 Feature selection statistics saved to: {RUN_DIR / 'feature_selection_statistics.csv'}")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"  Total features: {len(FEATURE_COLS)}")
print(f"  Features selected >50% of time: {sum(1 for f in frequencies if f > 50)}")
print(f"  Features selected >25% of time: {sum(1 for f in frequencies if f > 25)}")
print(f"  Features selected <10% of time: {sum(1 for f in frequencies if f < 10)}")
print(f"  Average selection frequency: {np.mean(frequencies):.1f}%")
print(f"  Median selection frequency: {np.median(frequencies):.1f}%")
